# 1. Read and inspect the data

Run cells in order. Queries and conversions are visible below. Each source read is saved as CSV under `../data/extract/sources/`; prepared tables are saved under `../data/extract/`.

Only this notebook needs database access. Tenant/environment and authentication come from the repository `.env`. Connections are read-only.

The input defines the scored collection, periods, peers and social rules. The forecast target does not change those periods.

In [ ]:
import os
from contextlib import ExitStack
from datetime import UTC, datetime
from pathlib import Path
from runpy import run_path

import pandas as pd
from aily_py_commons.aily_settings import AilySettings
from dotenv import load_dotenv

from aily_project.services.fmi.configuration import input_policy
from aily_project.services.fmi.connections import read_connections
from aily_project.services.fmi.extraction import (
    DOCUMENT_RELATION,
    SPECS,
    candidate_mappings,
    discover,
)
from aily_project.services.fmi.handover import candidate_symbols
from aily_project.services.fmi.inputs import read_inputs
from aily_project.services.fmi.issuer_measurements import ISSUERS, prepare_measurements
from aily_project.services.fmi.models import (
    fingerprint,
    instant,
    truth,
)
from aily_project.services.fmi.policy import Policy

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
FMI = ROOT / "scripts/fashion-momentum-index"
load_dotenv(ROOT / ".env", override=False)
INPUTS_PATH = Path(
    os.environ.get("FMI_INPUTS_PATH", str(FMI / "inputs/FMI_inputs.json"))
)
DATA = Path(os.environ.get("FMI_DATA_DIR", str(FMI / "data")))
OUT = Path(os.environ.get("FMI_OUTPUT_DIR", str(ROOT / "outputs/fmi")))

# Small file-reading, CSV-saving and read-only query helpers.
io = run_path(str(FMI / "helpers.py"))
load_json = io["load_json"]
save_table, load_table = io["save_table"], io["load_table"]
read_source, read_snapshot = io["read_source"], io["read_snapshot"]
begin_extract, publish_extract = io["begin_extract"], io["publish_extract"]

inputs = read_inputs(INPUTS_PATH)
policy = input_policy(INPUTS_PATH, Policy())
print("Input:", INPUTS_PATH)
print("Data:", DATA)
print("Delivery:", OUT)


EXTRACT = begin_extract(DATA)
SOURCES = EXTRACT / "sources"
query_settings = load_json(INPUTS_PATH)["query"]
FIRST_YEAR, LAST_YEAR = query_settings["first_year"], query_settings["last_year"]
ROW_LIMIT = query_settings["row_limit"]
assert 1900 <= FIRST_YEAR <= LAST_YEAR <= 2100
assert 1 <= ROW_LIMIT <= 500000
print("Bounded query years:", FIRST_YEAR, LAST_YEAR, "row cap:", ROW_LIMIT)
display(
    pd.DataFrame(inputs.entities).reindex(
        columns=["entity_id", "name", "light_code", "shared_symbol", "handle"]
    )
)

## 1.1 Review the requested financial periods

This is a parameter table, not downloaded revenue. Inspect subject, scope and all three roles before querying.

In [ ]:
requested_periods = []
for analysis in inputs.analyses:
    for entity_id, spec in (
        analysis["dossier"].get("financial", {}).get("entities", {}).items()
    ):
        for role in ("oldest", "earlier", "latest"):
            requested_periods.append(
                {
                    "analysis_id": analysis["analysis_id"],
                    "entity_id": entity_id,
                    "subject": spec.get("subject"),
                    "scope": spec.get("scope"),
                    "role": role,
                    **spec.get("periods", {}).get(role, {}),
                }
            )
display(save_table(requested_periods, EXTRACT / "requested-periods"))
save_table(
    [{"parameter": k, "value": v} for k, v in policy.model_dump().items()],
    EXTRACT / "policy",
)

## 1.2 Open read-only connections and discover columns

Settings initialize once here, before DAL. No cloud clients are created by imports.
If you stop the notebook early, run the closing cell in section 1.9.
A new run first closes any previous connection owned by this notebook.

In [ ]:
previous_connections = globals().get("connections")
if previous_connections is not None:
    previous_connections.close()
AilySettings(use_dotenv=False, verbose=False)
connections = ExitStack()
try:
    light, shared = connections.enter_context(read_connections())
except Exception:
    connections.close()
    raise

stamp = datetime.now(UTC).isoformat()
metadata = {
    "authenticated": False,
    "extracted_at": stamp,
    "first_year": FIRST_YEAR,
    "last_year": LAST_YEAR,
    "row_limit": ROW_LIMIT,
    "queries": [],
    "limitations": [],
    "resolved_policy": policy.model_dump(),
    "connection_context": {
        "tenant": os.environ.get("AILY_TENANT"),
        "environment": os.environ.get("AILY_ENV"),
        "light_aws_profile": os.environ.get("AWS_PROFILE"),
        "shared_aws_profile": os.environ.get("FMI_SHARED_AWS_PROFILE"),
    },
    "shared_role_warning": "Session read-only is not database-enforced "
    "read-only grants.",
}
light_schema = discover(light, [r for r in SPECS if r.startswith(("light.", "ai."))])
shared_schema = discover(
    shared, [r for r in SPECS if not r.startswith(("light.", "ai."))]
)
metadata["discovered_columns"] = {
    relation: sorted(columns)
    for relation, columns in (light_schema | shared_schema).items()
    if relation in SPECS
}
schema_rows = [
    {"relation": relation, "column": column}
    for relation, columns in metadata["discovered_columns"].items()
    for column in columns
]
display(save_table(schema_rows, EXTRACT / "schema"))

## 1.3 Company registry and query identifiers

Registry hits are search candidates, not new approvals. The scorer still uses the JSON's explicit subject mappings.
The queries also retrieve source aliases for inspection; they do not change parent/segment routing.

In [ ]:
registry = read_source(
    light,
    light_schema,
    "light.light_companies",
    "TRUE",
    (),
    {"company_code"},
    SOURCES,
    metadata,
    ROW_LIMIT,
)
mappings = candidate_mappings(inputs, registry)
metadata["candidate_mappings"] = mappings
display(save_table(mappings, EXTRACT / "identity-candidates"))

codes = sorted(
    {str(e["light_code"]) for e in inputs.entities if e.get("light_code")}
    | {str(r["company_code"]) for r in mappings}
    | {
        str(e["parent"]["light_code"])
        for e in inputs.entities
        if e.get("parent", {}).get("light_code")
    }
)
symbols = sorted(
    set(codes)
    | {str(e["shared_symbol"]) for e in inputs.entities if e.get("shared_symbol")}
    | {str(r.get("company_ticker") or r["company_code"]) for r in mappings}
    | {s for e in inputs.entities for s in candidate_symbols(e)}
    | {
        str(e["parent"].get("shared_symbol") or e["parent"].get("ticker"))
        for e in inputs.entities
        if e.get("parent", {}).get("shared_symbol") or e.get("parent", {}).get("ticker")
    }
    | {
        alias.strip()
        for e in inputs.entities
        if truth(e.get("mapping_approved")) and e.get("evidence")
        for alias in str(e.get("aliases") or "").split(";")
        if alias.strip()
    }
)
handles = sorted(
    {str(e["handle"]).lower().lstrip("@") for e in inputs.entities if e.get("handle")}
    | {
        str(m["handle"]).lower().lstrip("@")
        for a in inputs.analyses
        if a["dossier"].get("social", {}).get("platform") == "instagram"
        for m in a["dossier"]["social"].get("members", [])
        if m.get("handle")
    }
)


def slots(items):
    return ",".join(["%s"] * len(items))


save_table(
    [
        {"source": source, "identifier": value}
        for source, values in (
            ("light", codes),
            ("shared", symbols),
            ("instagram", handles),
        )
        for value in values
    ],
    EXTRACT / "query-identifiers",
)
print(
    len(codes),
    "Light codes;",
    len(symbols),
    "source search symbols;",
    len(handles),
    "Instagram handles",
)

## 1.4 Light actuals and forecasts

Inspect both source tables separately. Native fiscal labels are retained.
The source has no verified scope, multiplier or publication vintage; do not invent those fields.

In [ ]:
financials = []
for relation in (
    "light.light_financial_statements_actuals",
    "ai.light_financial_statements_forecast",
):
    rows = (
        read_source(
            light,
            light_schema,
            relation,
            f"company_code IN ({slots(codes)}) AND year BETWEEN %s AND %s "
            "AND type_filter_code IN "
            "('revenue','cost_of_revenue','gross_profit','ebitda')",
            (*codes, FIRST_YEAR, LAST_YEAR),
            {"company_code", "year", "type_filter_code", "type_filter_value"},
            SOURCES,
            metadata,
            ROW_LIMIT,
        )
        if codes
        else []
    )
    for row in rows:
        row.update(
            metric=row["type_filter_code"],
            amount=row["type_filter_value"],
            kind="forecast" if relation.startswith("ai.") else "actual",
            source_classification="current_snapshot",
            available_at=None,
            vintage_verified=False,
            period_label=f"Light native {row['year']} {row.get('quarter')}",
        )
    financials.extend(rows)
    display(pd.DataFrame(rows).head(20))
save_table(financials, EXTRACT / "light-financials")

## 1.5 Shared financial sources

Read the consolidated view **and** its underlying providers. Each observation stays separate.
Revenue, cost, gross profit and EBITDA become long-format rows with stable per-metric source IDs.
Reported, interpolated, estimated and consolidated candidates remain labelled differently.

In [ ]:
finance_sources = {
    "normalized.v_finance_financial_statements": "consolidated_requires_source_review",
    "normalized.finance_fmp_financial_statements": "reported",
    "normalized.finance_interpolated_financial_statements": "interpolated",
    "normalized.finance_perplexity_financial_statements": "estimated",
}
for relation, classification in finance_sources.items():
    rows = (
        read_source(
            shared,
            shared_schema,
            relation,
            f"symbol IN ({slots(symbols)}) AND calendar_year BETWEEN %s AND %s",
            (*symbols, FIRST_YEAR, LAST_YEAR),
            {"symbol", "calendar_year", "revenue"},
            SOURCES,
            metadata,
            ROW_LIMIT,
        )
        if symbols
        else []
    )
    display(pd.DataFrame(rows).head(10))
    for row in rows:
        captures = [
            t
            for c in ("accepted_date", "crawled_at")
            if (t := instant(row.get(c))) is not None
        ]
        for metric in ("revenue", "cost_of_revenue", "gross_profit", "ebitda"):
            if row.get(metric) is not None:
                financials.append(
                    {
                        **row,
                        "source_row_id": fingerprint(
                            {"row": row["source_row_id"], "metric": metric}
                        ),
                        "metric": metric,
                        "amount": row[metric],
                        "currency": row.get("reported_currency"),
                        "kind": "actual",
                        "source_classification": classification,
                        "period_label": (
                            f"Shared native {row['calendar_year']} {row.get('period')}"
                        ),
                        "available_at": max(captures).isoformat() if captures else None,
                        "vintage_verified": False,
                    }
                )
financial_table = save_table(financials, EXTRACT / "financials")
if not financial_table.empty:
    display(
        financial_table.groupby(["source_relation", "kind", "metric"])
        .size()
        .rename("rows")
    )
else:
    print("No financial candidates returned in this query scope.")

## 1.6 Correction evidence

The consolidated view already applies database corrections. Do not apply values twice.
Attach correction events and underlying source IDs for review. Corrections do not establish scope or historical vintage.

In [ ]:
override_scope = f"symbol IN ({slots(symbols)})"
if codes:
    override_scope += f" OR company_code IN ({slots(codes)})"
overrides = (
    read_source(
        shared,
        shared_schema,
        "denormalized.finance_financial_statement_overrides",
        f"({override_scope}) AND calendar_year BETWEEN %s AND %s",
        (*symbols, *codes, FIRST_YEAR, LAST_YEAR),
        {"symbol", "calendar_year", "metric", "changed_at"},
        SOURCES,
        metadata,
        ROW_LIMIT,
    )
    if symbols
    else []
)
display(pd.DataFrame(overrides).head(20))

aliases = {}
for company in registry:
    symbol = str(company.get("company_ticker") or company["company_code"])
    aliases.setdefault(symbol, set()).add(str(company["company_code"]))
source_index = {}
for row in financials:
    key = tuple(row.get(k) for k in ("symbol", "calendar_year", "period", "metric"))
    source_index.setdefault(key, []).append(row)
for row in financials:
    events = [
        e
        for e in overrides
        if (
            (row.get("symbol") and e.get("symbol") == row.get("symbol"))
            or (
                not e.get("symbol")
                and str(e.get("company_code"))
                in aliases.get(str(row.get("symbol")), set())
            )
        )
        and all(e.get(k) == row.get(k) for k in ("calendar_year", "period", "metric"))
    ]
    row["correction_events"] = events
    if row["source_classification"] != "consolidated_requires_source_review":
        continue
    key = tuple(row.get(k) for k in ("symbol", "calendar_year", "period", "metric"))
    row["source_candidates"] = [
        {
            k: r.get(k)
            for k in (
                "source_row_id",
                "source_relation",
                "amount",
                "currency",
                "source_classification",
                "available_at",
            )
        }
        for r in source_index[key]
        if r["source_relation"] != row["source_relation"]
    ]
    if events:
        exact = [e for e in events if e.get("symbol") == row.get("symbol")]
        latest = max(
            exact or events,
            key=lambda e: (
                str(e.get("changed_at") or ""),
                str(e.get("event_id") or ""),
            ),
        )
        if (
            latest.get("action") == "set"
            and row.get("currency")
            and latest.get("reported_currency") == row["currency"]
        ):
            row["source_classification"] = "corrected_requires_vintage_review"
            change_time, native_time = (
                instant(latest.get("changed_at")),
                instant(row.get("available_at")),
            )
            row["available_at"] = (
                max(change_time, native_time).isoformat()
                if change_time and native_time
                else None
            )
save_table(overrides, EXTRACT / "overrides")
save_table(financials, EXTRACT / "financials");

## 1.7 Instagram captures and profile history

Do not deduplicate yet: notebook 2 applies the dossier capture cutoff before deduplication.
Use the raw archive only for accounts absent from normalization, never merge mirror captures for the same account.

In [ ]:
post_relation = (
    "normalized.v_social_media_instagram_posts"
    if {"crawled_at", "post_id", "profile_username", "date_posted"}
    <= shared_schema.get("normalized.v_social_media_instagram_posts", set())
    else "normalized.social_media_instagram_posts"
)
posts = (
    read_source(
        shared,
        shared_schema,
        post_relation,
        f"LOWER(profile_username) IN ({slots(handles)}) "
        "AND date_posted >= %s AND date_posted < %s",
        (*handles, f"{FIRST_YEAR}-01-01", f"{LAST_YEAR + 1}-01-01"),
        {"profile_username", "date_posted", "post_id", "crawled_at"},
        SOURCES,
        metadata,
        ROW_LIMIT,
    )
    if handles
    else []
)
present_handles = {str(r["profile_username"]).lower() for r in posts}
missing_handles = [h for h in handles if h not in present_handles]
if missing_handles:
    fallback = read_source(
        shared,
        shared_schema,
        "denormalized.social_media_bright_data_instagram_posts",
        f"LOWER(user_posted) IN ({slots(missing_handles)}) "
        "AND date_posted >= %s AND date_posted < %s",
        (*missing_handles, f"{FIRST_YEAR}-01-01", f"{LAST_YEAR + 1}-01-01"),
        {"user_posted", "post_id", "date_posted", "crawled_at"},
        SOURCES,
        metadata,
        ROW_LIMIT,
    )
    for row in fallback:
        row.update(
            profile_username=row.get("user_posted"),
            profile_id=row.get("user_posted_id"),
            post_url=row.get("url"),
            caption=row.get("description"),
            post_type=row.get("content_type"),
            like_count=row.get("likes"),
            comment_count=row.get("num_comments"),
        )
    posts.extend(fallback)
post_table = save_table(posts, EXTRACT / "posts")
display(post_table.head(20))
if not post_table.empty:
    display(post_table.groupby("profile_username").size().rename("captures"))
else:
    print("No Instagram captures returned in this query scope.")

In [ ]:
profiles = (
    read_source(
        shared,
        shared_schema,
        "normalized.social_media_instagram_profiles",
        f"LOWER(profile_username) IN ({slots(handles)}) "
        "AND crawled_date >= %s AND crawled_date < %s",
        (*handles, f"{FIRST_YEAR}-01-01", f"{LAST_YEAR + 1}-01-01"),
        {"profile_username", "crawled_date"},
        SOURCES,
        metadata,
        ROW_LIMIT,
    )
    if handles
    else []
)
for row in profiles:
    values = [
        row[c]
        for c in ("followers_count", "followers", "follower_count")
        if row.get(c) is not None
    ]
    row["followers_count"] = values[0] if len(set(values)) == 1 else None
present_profiles = {str(r["profile_username"]).lower() for r in profiles}
missing_profiles = [h for h in handles if h not in present_profiles]
if missing_profiles:
    fallback = read_source(
        shared,
        shared_schema,
        "denormalized.social_media_bright_data_instagram_profiles",
        f"LOWER(account) IN ({slots(missing_profiles)}) "
        "AND crawled_date >= %s AND crawled_date < %s",
        (*missing_profiles, f"{FIRST_YEAR}-01-01", f"{LAST_YEAR + 1}-01-01"),
        {"account", "crawled_date"},
        SOURCES,
        metadata,
        ROW_LIMIT,
    )
    for row in fallback:
        row.update(
            profile_username=row.get("account"),
            profile_id=row.get("id"),
            followers_count=row.get("followers"),
        )
    profiles.extend(fallback)
display(save_table(profiles, EXTRACT / "profiles").head(20))

## 1.8 Full-source-document coverage

Full issuer disclosures exist in shared DB even when structured rows lack semantics.
Index their metadata separately. Dates in this index can disagree with the document body.
An index hit never supplies a scoring amount, scope or multiplier.

In [ ]:
document_codes = sorted(set(codes) | set(symbols) | ISSUERS)
documents = (
    read_source(
        shared,
        shared_schema,
        DOCUMENT_RELATION,
        f"company_code IN ({slots(document_codes)}) "
        "AND period_end >= %s AND period_end < %s",
        (*document_codes, f"{FIRST_YEAR}-01-01", f"{LAST_YEAR + 1}-01-01"),
        {"company_code", "period_end", "url"},
        SOURCES,
        metadata,
        ROW_LIMIT,
    )
    if document_codes
    else []
)
metadata["financial_source_documents"] = documents
display(save_table(documents, EXTRACT / "source-documents").head(20))

# Read full bodies only for supported issuer tables. Never use dossier amounts.
issuer_codes = sorted(ISSUERS)
issuer_documents = read_source(
    shared,
    shared_schema,
    DOCUMENT_RELATION,
    f"company_code IN ({slots(issuer_codes)}) "
    "AND period_end >= %s AND period_end < %s "
    "AND (report_type IN ('press release','annual report') "
    "OR document_type IN ('presentation','press release')) "
    "AND LENGTH(content::text) <= %s",
    (*issuer_codes, f"{FIRST_YEAR}-01-01", f"{LAST_YEAR + 1}-01-01", 500000),
    {"company_code", "period_end", "url", "content"},
    EXTRACT / "issuer-sources",
    metadata,
    1000,
    include_content=True,
)
measurements = prepare_measurements(issuer_documents)
financials.extend(measurements)
save_table(measurements, EXTRACT / "issuer-measurements")
save_table(financials, EXTRACT / "financials")
print(len(measurements), "source-decoded financial observations")
metadata["competitor_candidates"] = (
    read_source(
        shared,
        shared_schema,
        "normalized.finance_general_competitors",
        f"company_code IN ({slots(codes)})",
        tuple(codes),
        {"company_code", "competitor_code"},
        SOURCES,
        metadata,
        ROW_LIMIT,
    )
    if codes
    else []
)
print(
    "Competitor candidates are context; the assigned peer list stays in the input JSON."
)

### Optional: inspect a document body

Leave this off for the normal run. Choose a company code from the index above.
This bounded text download is for checking source semantics, not a financial parser.
A truncated body remains flagged. Read-only settings were verified before opening these sessions.

## 1.9 Close sessions, inspect coverage and save the handoff

A failed extraction does not replace the last complete snapshot.
Missing metadata is shown separately from returned measurement counts.

In [ ]:
connections.close()
metadata["limitations"].extend(
    [
        (
            "Financial scope, units and discrete reporting periods "
            "require source evidence."
        ),
        "Document metadata is inventory, not proof of a source measurement.",
        (
            "Light forecast publication vintage is unavailable; "
            "this run is not historical replay."
        ),
    ]
)
metadata["authenticated"] = True
save_table(registry, EXTRACT / "registry")
save_table(metadata["competitor_candidates"], EXTRACT / "competitor-candidates")
save_table(
    [{"limitation": text} for text in metadata["limitations"]], EXTRACT / "limitations"
)
save_table(
    [
        {
            "authenticated": True,
            "extracted_at": stamp,
            "first_year": FIRST_YEAR,
            "last_year": LAST_YEAR,
            "financial_rows": len(financials),
            "instagram_captures": len(posts),
        }
    ],
    EXTRACT / "metadata",
)
display(pd.DataFrame(metadata["queries"])[["source_relation", "row_count", "columns"]])
semantics = [
    {
        "field": field,
        "missing_rows": sum(r.get(field) in (None, "") for r in financials),
        "total_rows": len(financials),
    }
    for field in (
        "scope",
        "currency",
        "unit_multiplier",
        "period_start",
        "period_end",
        "period_type",
    )
]
display(save_table(semantics, EXTRACT / "financial-semantics"))
print("\n".join(metadata["limitations"]))
publish_extract(EXTRACT, DATA / "extract")
EXTRACT = DATA / "extract"
print("Next: 02_score.ipynb reads the CSV tables in", EXTRACT)